In [ ]:
# ── Colab 전용 설정 ───────────────────────────────────────────
# Chrome 브라우저 + ChromeDriver 설치
!apt-get update -q
!apt-get install -y -q chromium-browser chromium-chromedriver
!pip install selenium -q

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,986 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,889 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,917 kB]

In [ ]:
# ── 1단계: 실제 경로 확인 ─────────────────────────────────────
!which chromium-browser
!which chromium
!which chromedriver
!find / -name "chromedriver" 2>/dev/null

/usr/bin/chromium-browser
/usr/bin/chromedriver
/usr/bin/chromedriver
/usr/lib/chromium-browser/chromedriver


In [ ]:
# ── 설치 확인 ─────────────────────────────────────────────────
!apt-get install -y chromium-browser chromium-chromedriver 2>&1 | tail -5
!ls /usr/bin/chrom*
!ls /usr/lib/chrom*

Building dependency tree...
Reading state information...
chromium-browser is already the newest version (1:85.0.4183.83-0ubuntu2.22.04.1).
chromium-chromedriver is already the newest version (1:85.0.4183.83-0ubuntu2.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 15 not upgraded.
/usr/bin/chromedriver  /usr/bin/chromium-browser
chromedriver


In [ ]:
!pip install webdriver-manager -q

In [ ]:
# chromium 기존 버전 제거 후 최신으로 재설치
!apt-get remove -y chromium-browser chromium-chromedriver -q
!apt-get install -y -q wget curl unzip

# Google Chrome 최신 stable 설치 (chromium 대신)
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q

# 설치된 버전 확인
!google-chrome --version

Reading package lists...
Building dependency tree...
Reading state information...
The following packages were automatically installed and are no longer required:
  apparmor libfuse3-3 snapd squashfs-tools systemd-hwe-hwdb udev
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  chromium-browser chromium-chromedriver
0 upgraded, 0 newly installed, 2 to remove and 15 not upgraded.
After this operation, 243 kB disk space will be freed.
(Reading database ... 118709 files and directories currently installed.)
Removing chromium-chromedriver (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Removing chromium-browser (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...
Reading package lists...
Building dependency tree...
Reading state information...
curl is already the newest version (7.81.0-1ubuntu1.23).
unzip is already the newest version (6.0-26ubuntu3.2).
wget is already the newes

In [ ]:
import time
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

# 이번엔 google-chrome 실행 파일 경로로 변경
options.binary_location = "/usr/bin/google-chrome"

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)
print("드라이버 초기화 성공!")

드라이버 초기화 성공!


In [ ]:
# ── 셀렉터 테스트 (본 크롤링 전에 꼭 실행) ──────────────────────
BASE_LIST_URL = (
    "https://www.paulaschoice.co.kr/ingredients"
    "?csortb1=name&csortd1=1"
    "&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8"
    "&sz=10&start=0"
)

driver.get(BASE_LIST_URL)
time.sleep(3)  # 페이지 완전 로딩 대기

# ── 테스트 1: 성분 링크가 잡히는지 확인 ──────────────────────────
links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/ingredients/ingredient-']")
print(f"[링크 수집] {len(links)}개 발견")
for el in links[:3]:  # 처음 3개만 출력
    print(" ", el.get_attribute("href"))

# ── 테스트 2: 첫 번째 상세 페이지 접속해서 셀렉터 확인 ─────────────
if links:
    first_url = links[0].get_attribute("href")
    driver.get(first_url)
    time.sleep(2)

    # 성분 이름
    try:
        name = driver.find_element(By.CSS_SELECTOR, "h1").text.strip()
        print(f"\n[성분명] {name}")
    except:
        print("\n[성분명] 셀렉터 못 찾음 → h1 확인 필요")

    # 등급
    try:
        rating = driver.find_element(
            By.CSS_SELECTOR, ".ingredient-rating, .rating-label"
        ).text.strip()
        print(f"[등급] {rating}")
    except:
        print("[등급] 셀렉터 못 찾음 → 수정 필요")

    # 본문
    try:
        paragraphs = driver.find_elements(
            By.CSS_SELECTOR, ".ingredient-description p, .content-body p"
        )
        desc = "\n".join(p.text.strip() for p in paragraphs if p.text.strip())
        print(f"[본문 미리보기]\n{desc[:300]}")  # 300자만 출력
    except:
        print("[본문] 셀렉터 못 찾음 → 수정 필요")

[링크 수집] 10개 발견
  https://www.paulaschoice.co.kr/ingredients/ingredient-almond-borage-linseed-olive-acids-glycerides.html?csortb1=name&csortd1=1&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8&sz=10&fdid=ingredients&start=0
  https://www.paulaschoice.co.kr/ingredients/ingredient-10-hydroxydecanoic-acid.html?csortb1=name&csortd1=1&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8&sz=10&fdid=ingredients&start=0
  https://www.paulaschoice.co.kr/ingredients/ingredient-3-o-ethyl-ascorbic-acid.html?csortb1=name&csortd1=1&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8&sz=10&fdid=ingredients&start=0

[성분명] 셀렉터 못 찾음 → h1 확인 필요
[등급] 셀렉터 못 찾음 → 수정 필요
[본문 미리보기]



In [ ]:
# ── 상세 페이지 HTML 구조 확인 ────────────────────────────────
detail_url = "https://www.paulaschoice.co.kr/ingredients/ingredient-almond-borage-linseed-olive-acids-glycerides.html?csortb1=name&csortd1=1&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8&sz=10&fdid=ingredients&start=0"

driver.get(detail_url)
time.sleep(2)

# 페이지 전체 텍스트 출력 (어디에 성분명/등급/본문이 있는지 확인)
body_text = driver.find_element(By.TAG_NAME, "body").text
print(body_text[:3000])  # 앞 3000자만 출력

[이달의 성분] 피부가 먹는 단백질, 펩타이드
[공식몰 단독] 10%나이아신 듀오세트
[ONLY 신규회원] 최대 56,000원 상당 혜택
0
제품 카테고리
제품 라인별
신규&멤버십혜택
공식몰 이벤트
리얼리뷰
뷰티피디아
스킨케어 성분사전/검색결과
(아몬드/보리지/아마/올리브)애씨드/글리세라이즈
Almond/Borage/Linseed/Olive Acids/Glycerides*
등급: 훌륭함
효과별: 수분 공급, 안티에이징
분류: 유연제(연화제), 항산화제
(아몬드/보리지/아마/올리브)애씨드/글리세라이즈 성분소개
아몬드, 보리지, 아마, 올리브 등 무향의 식물성 오일에서 비롯된 트라이글리세라이드와 지방산의 혼합물. 이 혼합물은 피부에 수분을 공급하고 보충하며 항산화 효과를 제공하고 피부 본연의 회복력을 향상할 수 있다.
*대한화장품성분사전에 등록된 표준화 국문, 영문명칭
연관성분: 아몬드오일, 보리지씨오일
검색결과로 돌아가기
(아몬드/보리지/아마/올리브)애씨드/글리세라이즈 참고논문
American Journal of Clinical Dermatology, February 2018, pages 103-117
International Journal of Molecular Sciences, December 2017, ePublication
Journal of Cosmetic Dermatology, December 2016, pages 549-558
OUR COLLECTIONS
스킨 퍼펙팅
유스 익스텐딩
플럼프+펌
브라이튼 업
베리어 밸런싱
클리어
스킨 쉴드+
바디 리바이벌
WHO WE ARE
폴라초이스의 진실된 철학
폴라비가운 스토리
과학 자문 위원회
연구&교육팀
지속 가능한 지구환경을 위한 우리의 노력
전세계 폴라초이스
채용정보
MEMBERSHIP
신규&멤버십혜택
CLIENT SERVICES
뷰티컨설턴트
FAQ
공지사항
이메일 문의
고객센터 : 1661-6656
월~금: 오전 10시 ~ 오후 5시, 점심시간 : 12시 ~ 1시
(주말, 법정공휴일 휴무)
이용약관개인정보

In [ ]:
# ── 실제 셀렉터 탐색 ──────────────────────────────────────────
driver.get(detail_url)
time.sleep(2)

# 성분명 (한글명) 찾기
print("=== h1, h2, h3 태그 전부 출력 ===")
for tag in ["h1", "h2", "h3"]:
    els = driver.find_elements(By.TAG_NAME, tag)
    for el in els:
        txt = el.text.strip()
        if txt:
            print(f"<{tag}> : {txt[:80]}")

print("\n=== 등급/효과별/분류 포함된 요소 찾기 ===")
# "등급" 텍스트 근처 요소 탐색
els = driver.find_elements(By.XPATH, "//*[contains(text(), '등급')]")
for el in els:
    print(f"태그: <{el.tag_name}> 클래스: {el.get_attribute('class')} | 텍스트: {el.text.strip()[:100]}")

print("\n=== 본문 설명 포함된 요소 찾기 ===")
# "성분소개" 텍스트 근처 탐색
els = driver.find_elements(By.XPATH, "//*[contains(text(), '성분소개')]")
for el in els:
    print(f"태그: <{el.tag_name}> 클래스: {el.get_attribute('class')} | 텍스트: {el.text.strip()[:200]}")

=== h1, h2, h3 태그 전부 출력 ===
<h3> : Almond/Borage/Linseed/Olive Acids/Glycerides*

=== 등급/효과별/분류 포함된 요소 찾기 ===
태그: <span> 클래스: large7 normalcase ko_style | 텍스트: 등급:

=== 본문 설명 포함된 요소 찾기 ===
태그: <div> 클래스: IngredientPagestyles__DescriptionTitle-sc-rgh8tc-15 clFMpc large3 normalcase clip ko_style | 텍스트: (아몬드/보리지/아마/올리브)애씨드/글리세라이즈 성분소개


In [ ]:
# ── 등급 값이랑 본문 텍스트 정확히 찾기 ──────────────────────
driver.get(detail_url)
time.sleep(2)

# 1. 등급 값 ("훌륭함") 찾기 - "등급:" span의 부모/형제 요소 탐색
print("=== 등급 관련 요소 ===")
rating_label = driver.find_element(By.XPATH, "//span[contains(text(), '등급:')]")
parent = rating_label.find_element(By.XPATH, "..")  # 부모 요소
print(f"부모 태그: <{parent.tag_name}> 클래스: {parent.get_attribute('class')}")
print(f"부모 전체 텍스트: {parent.text.strip()[:200]}")

print("\n=== 효과별/분류 포함 요소 ===")
els = driver.find_elements(By.XPATH, "//*[contains(text(), '효과별')]")
for el in els:
    print(f"태그: <{el.tag_name}> 클래스: {el.get_attribute('class')} | {el.text.strip()[:100]}")

# 2. 본문 설명 텍스트 찾기 - "성분소개" div의 형제/자식 탐색
print("\n=== 본문 설명 텍스트 ===")
desc_title = driver.find_element(
    By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
)
# 형제 요소 (바로 다음 sibling) 탐색
siblings = driver.find_elements(
    By.XPATH, "//div[contains(@class, 'DescriptionTitle')]/following-sibling::*[1]"
)
for s in siblings:
    print(f"태그: <{s.tag_name}> 클래스: {s.get_attribute('class')}")
    print(f"텍스트: {s.text.strip()[:300]}")

=== 등급 관련 요소 ===


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//span[contains(text(), '등급:')]"}
  (Session info: chrome=147.0.7727.55); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
#0 0x55ac111e8b6a <unknown>
#1 0x55ac10bea265 <unknown>
#2 0x55ac10c3cf76 <unknown>
#3 0x55ac10c3d1b1 <unknown>
#4 0x55ac10c887d4 <unknown>
#5 0x55ac10c85969 <unknown>
#6 0x55ac10c305cf <unknown>
#7 0x55ac10c31391 <unknown>
#8 0x55ac111ae04b <unknown>
#9 0x55ac111b100d <unknown>
#10 0x55ac1119a808 <unknown>
#11 0x55ac111b1ba0 <unknown>
#12 0x55ac11181280 <unknown>
#13 0x55ac111d5db8 <unknown>
#14 0x55ac111d5f88 <unknown>
#15 0x55ac111e75de <unknown>
#16 0x7b575f23cac3 <unknown>


In [ ]:
# ── 페이지 소스에서 "등급" 주변 HTML 직접 확인 ────────────────
import re

driver.get(detail_url)
time.sleep(2)

html = driver.page_source  # 전체 HTML 소스

# "등급" 단어 주변 200자씩 잘라서 출력
matches = [m.start() for m in re.finditer("등급", html)]
print(f"'등급' 등장 횟수: {len(matches)}\n")

for i, pos in enumerate(matches[:5]):  # 최대 5개만
    snippet = html[max(0, pos-100) : pos+200]
    print(f"--- [{i+1}번째] ---")
    print(snippet)
    print()

'등급' 등장 횟수: 5

--- [1번째] ---
redientPagestyles__RatingWrapper-sc-rgh8tc-7 dchpDi aICZT"><span class="large7 normalcase ko_style">등급:</span> <span class="ColoredIngredientRating__Rating-sc-r02772-0 bKTprm large6 uppercase ko_style">훌륭함</span></div><div class="IngredientPagestyles__RowWrapper-sc-rgh8tc-6 IngredientPagestyles__Ben

--- [2번째] ---
gt;\r\n&lt;!--&lt;p&gt;&lt;a href=\&quot;https://www.paulaschoice.co.kr/membership-page\&quot;&gt;회원등급 안내&lt;/a&gt;&lt;/p&gt;--&gt;\r\n&lt;!--&lt;p&gt;&lt;a href=\&quot;https://www.paulaschoice.co.kr/bestreviewer\&quot;&gt;베스트 리뷰어&lt;/a&gt;&lt;/p&gt;--&gt;&quot;,&quot;html4&quot;:&quot;&lt;!--p&gt;&

--- [3번째] ---
품 입니다.&quot;,&quot;jarPackagingModalButtonText&quot;:&quot;네, 확인했어요.&quot;,&quot;rating&quot;:&quot;등급&quot;,&quot;benefits&quot;:&quot;효과별&quot;,&quot;atAGlance&quot;:&quot;[name] 한눈에 보기&quot;,&quot;description&quot;:&quot;[name] 성분소개&quot;,&quot;related&quot;:&quot;연관성분&quot;,&quot;productsWith&qu

--- [4번째] ---
uot;:&quot;참고문헌&quot;,&

In [ ]:
# ── 셀렉터 최종 확인 테스트 ───────────────────────────────────
driver.get(detail_url)
time.sleep(2)

# 한글 성분명 (h2 또는 상단 텍스트)
print("=== 성분명 (한글) ===")
try:
    # "성분소개" 타이틀에서 성분명 추출
    desc_title = driver.find_element(By.XPATH, "//div[contains(@class, 'DescriptionTitle')]")
    # "성분소개" 앞부분만 잘라내기
    full_text = desc_title.text.strip()
    ko_name = full_text.replace("성분소개", "").strip()
    print(ko_name)
except:
    print("못 찾음")

# 영문 성분명
print("\n=== 성분명 (영문) ===")
try:
    en_name = driver.find_element(By.TAG_NAME, "h3").text.strip()
    print(en_name)
except:
    print("못 찾음")

# 등급
print("\n=== 등급 ===")
try:
    rating = driver.find_element(
        By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
    ).text.strip()
    print(rating)
except:
    print("못 찾음")

# 효과별
print("\n=== 효과별 ===")
try:
    els = driver.find_elements(By.XPATH, "//*[contains(text(), '효과별')]/following-sibling::*")
    for el in els[:3]:
        print(el.text.strip())
except:
    print("못 찾음")

# 본문 설명
print("\n=== 본문 설명 ===")
try:
    desc_div = driver.find_element(
        By.XPATH, "//div[contains(@class, 'DescriptionTitle')]/following-sibling::div[1]"
    )
    print(desc_div.text.strip()[:300])
except:
    print("못 찾음")

=== 성분명 (한글) ===
(아몬드/보리지/아마/올리브)애씨드/글리세라이즈

=== 성분명 (영문) ===


=== 등급 ===
훌륭함

=== 효과별 ===

=== 본문 설명 ===
아몬드, 보리지, 아마, 올리브 등 무향의 식물성 오일에서 비롯된 트라이글리세라이드와 지방산의 혼합물. 이 혼합물은 피부에 수분을 공급하고 보충하며 항산화 효과를 제공하고 피부 본연의 회복력을 향상할 수 있다.
*대한화장품성분사전에 등록된 표준화 국문, 영문명칭


In [ ]:
# ── 추가 필드 셀렉터 탐색 ─────────────────────────────────────
driver.get(detail_url)
time.sleep(2)

import re

html = driver.page_source

# 효과별, 분류, 연관성분, 참고논문 주변 HTML 확인
for keyword in ["효과별", "분류", "연관성분", "참고논문"]:
    matches = [m.start() for m in re.finditer(keyword, html)]
    print(f"\n{'='*10} '{keyword}' ({len(matches)}번 등장) {'='*10}")
    for pos in matches[:2]:  # 최대 2개만
        snippet = html[max(0, pos-150) : pos+300]
        print(snippet)
        print()


========== '효과별' (2번 등장) ==========
class="IngredientPagestyles__RowWrapper-sc-rgh8tc-6 IngredientPagestyles__Benefits-sc-rgh8tc-8 dchpDi bPjLse"><div class="large7 normalcase ko_style">효과별: <a class="Link__StyledLink-sc-wqxkev-0 djGulS" href="/ingredients?crefn1=ingredientBenefits&amp;crefv1=%EC%88%98%EB%B6%84%20%EA%B3%B5%EA%B8%89"><span class="HyperLink-sc-2q4nbr-0 IngredientPagestyles__BlockLink-sc-rgh8tc-9 hswSYV euxtRI">수분 공급</span></a>, <a class="Link__StyledLink-sc-wqxkev-0 

단지(Jar)형 패키지 제품 입니다.&quot;,&quot;jarPackagingModalButtonText&quot;:&quot;네, 확인했어요.&quot;,&quot;rating&quot;:&quot;등급&quot;,&quot;benefits&quot;:&quot;효과별&quot;,&quot;atAGlance&quot;:&quot;[name] 한눈에 보기&quot;,&quot;description&quot;:&quot;[name] 성분소개&quot;,&quot;related&quot;:&quot;연관성분&quot;,&quot;productsWith&quot;:&quot;연관제품&quot;,&quot;backToIngredientDictionary&quot;:&quot;검색결과로 돌아가기&quot;,&quot;referencesTitle&quot;:&quot;[name] 참고논문&quot;,&


========== '분류' (3번 등장) ==========
9 hswSYV euxtRI">안티에이징<

In [ ]:
# ── 테스트: 첫 페이지 10개만 수집 ────────────────────────────
results = []

list_url = (
    "https://www.paulaschoice.co.kr/ingredients"
    "?csortb1=name&csortd1=1"
    "&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8"
    "&sz=10&start=0"
)

driver.get(list_url)
time.sleep(2)

ingredient_links = get_ingredient_links()
print(f"링크 {len(ingredient_links)}개 수집\n")

for idx, link in enumerate(ingredient_links):
    print(f"[{idx+1}/10] 수집 중...")
    detail_data = scrape_detail_page(link)
    results.append(detail_data)
    driver.get(list_url)
    time.sleep(1.5)

print("\n── 수집 결과 미리보기 ──")
for i, r in enumerate(results):
    print(f"\n[{i+1}번째 성분]")
    for k, v in r.items():
        print(f"  {k}: {v[:60] if v else '(없음)'}")

링크 10개 수집

[1/10] 수집 중...
[2/10] 수집 중...
[3/10] 수집 중...
[4/10] 수집 중...
[5/10] 수집 중...
[6/10] 수집 중...
[7/10] 수집 중...
[8/10] 수집 중...
[9/10] 수집 중...
[10/10] 수집 중...

── 수집 결과 미리보기 ──

[1번째 성분]
  url: https://www.paulaschoice.co.kr/ingredients/ingredient-almond
  name: (없음)
  rating: (없음)
  description: (없음)

[2번째 성분]
  url: https://www.paulaschoice.co.kr/ingredients/ingredient-10-hyd
  name: (없음)
  rating: (없음)
  description: (없음)

[3번째 성분]
  url: https://www.paulaschoice.co.kr/ingredients/ingredient-3-o-et
  name: (없음)
  rating: (없음)
  description: (없음)

[4번째 성분]
  url: https://www.paulaschoice.co.kr/ingredients/ingredient-c10-30
  name: (없음)
  rating: (없음)
  description: (없음)

[5번째 성분]
  url: https://www.paulaschoice.co.kr/ingredients/ingredient-c-13-1
  name: (없음)
  rating: (없음)
  description: (없음)

[6번째 성분]
  url: https://www.paulaschoice.co.kr/ingredients/ingredient-d-alph
  name: (없음)
  rating: (없음)
  description: (없음)

[7번째 성분]
  url: https://www.paulaschoice.co.kr/ingredients/ingr

In [ ]:
# ── scrape_detail_page 함수 재정의 ───────────────────────────
from selenium.common.exceptions import NoSuchElementException

def scrape_detail_page(detail_url):
    driver.get(detail_url)
    time.sleep(1.5)

    data = {}

    # 한글명
    try:
        desc_title = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
        )
        data["한글명"] = desc_title.text.replace("성분소개", "").strip()
    except NoSuchElementException:
        data["한글명"] = ""

    # 영문명
    try:
        data["영문명"] = driver.find_element(By.TAG_NAME, "h3").text.strip()
    except NoSuchElementException:
        data["영문명"] = ""

    # 등급
    try:
        data["등급"] = driver.find_element(
            By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
        ).text.strip()
    except NoSuchElementException:
        data["등급"] = ""

    # 효과별
    try:
        benefits_div = driver.find_element(By.CSS_SELECTOR, "div[class*='Benefits']")
        benefit_links = benefits_div.find_elements(By.TAG_NAME, "a")
        data["효과별"] = ", ".join(a.text.strip() for a in benefit_links if a.text.strip())
    except NoSuchElementException:
        data["효과별"] = ""

    # 분류
    try:
        category_div = driver.find_element(
            By.XPATH, "//div[contains(text(), '분류:')]"
        )
        category_links = category_div.find_elements(By.TAG_NAME, "a")
        data["분류"] = ", ".join(a.text.strip() for a in category_links if a.text.strip())
    except NoSuchElementException:
        data["분류"] = ""

    # 성분설명
    try:
        desc_body = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]/following-sibling::div[1]"
        )
        data["성분설명"] = desc_body.text.strip()
    except NoSuchElementException:
        data["성분설명"] = ""

    # 연관성분
    try:
        related_div = driver.find_element(By.CSS_SELECTOR, "div[class*='RelatedWrapper']")
        related_links = related_div.find_elements(By.TAG_NAME, "a")
        data["연관성분"] = ", ".join(a.text.strip() for a in related_links if a.text.strip())
    except NoSuchElementException:
        data["연관성분"] = ""

    # 참고논문
    try:
        ref_divs = driver.find_elements(By.CSS_SELECTOR, "div[class*='__Reference-']")
        data["참고논문"] = "\n".join(d.text.strip() for d in ref_divs if d.text.strip())
    except NoSuchElementException:
        data["참고논문"] = ""

    return data

print("함수 재정의 완료!")

함수 재정의 완료!


In [ ]:
# ── 단일 성분으로 바로 확인 ───────────────────────────────────
test_url = "https://www.paulaschoice.co.kr/ingredients/ingredient-almond-borage-linseed-olive-acids-glycerides.html?csortb1=name&csortd1=1&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8&sz=10&fdid=ingredients&start=0"

result = scrape_detail_page(test_url)

for k, v in result.items():
    print(f"{k}: {v[:80] if v else '(없음)'}")

한글명: (아몬드/보리지/아마/올리브)애씨드/글리세라이즈
영문명: (없음)
등급: 훌륭함
효과별: 수분 공급, 안티에이징
분류: (없음)
성분설명: 아몬드, 보리지, 아마, 올리브 등 무향의 식물성 오일에서 비롯된 트라이글리세라이드와 지방산의 혼합물. 이 혼합물은 피부에 수분을 공급하고 보충
연관성분: 아몬드오일, 보리지씨오일
참고논문: American Journal of Clinical Dermatology, February 2018, pages 103-117
Internati


In [ ]:
# ── 영문명, 분류 셀렉터 탐색 ──────────────────────────────────
import re

driver.get(test_url)
time.sleep(2)

html = driver.page_source

# 영문명 주변 HTML 확인 ("Almond" 텍스트 근처)
print("=== 영문명 주변 HTML ===")
matches = [m.start() for m in re.finditer("Almond/Borage", html)]
for pos in matches[:2]:
    print(html[max(0, pos-200) : pos+200])
    print()

# 분류 주변 HTML 확인
print("=== 분류 주변 HTML ===")
matches = [m.start() for m in re.finditer("유연제", html)]
for pos in matches[:2]:
    print(html[max(0, pos-200) : pos+200])
    print()

=== 영문명 주변 HTML ===
과</a></div><div class="IngredientPagestyles__Name-sc-rgh8tc-5 cehCvR large1 normalcase clip ko_style">(아몬드/보리지/아마/올리브)애씨드/글리세라이즈</div><h3 class="IngredientPagestyles__EnglishName-sc-rgh8tc-27 fnDVHp">Almond/Borage/Linseed/Olive Acids/Glycerides*</h3><div class="IngredientPagestyles__RowWrapper-sc-rgh8tc-6 IngredientPagestyles__RatingWrapper-sc-rgh8tc-7 dchpDi aICZT"><span class="large7 normalcase 

,&quot;type&quot;:&quot;ingredient&quot;,&quot;title&quot;:&quot;(아몬드/보리지/아마/올리브)애씨드/글리세라이즈 | 폴라초이스 공식 온라인몰&quot;,&quot;name&quot;:&quot;(아몬드/보리지/아마/올리브)애씨드/글리세라이즈&quot;,&quot;englishName&quot;:&quot;Almond/Borage/Linseed/Olive Acids/Glycerides*&quot;,&quot;backToResultsUrl&quot;:&quot;/ingredients?csortb1=name&amp;csortd1=1&amp;crefn1=ingredientRating&amp;crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8&amp;sz

=== 분류 주변 HTML ===
ents?crefn1=ingredientCategories&amp;crefv1=%EC%9C%A0%EC%97%B0%EC%A0%9C%28%EC%97%B0%ED%99%94%EC%A0%9C%29"><span class="HyperLink-sc-2q4nbr-0 IngredientPagest

In [ ]:
def scrape_detail_page(detail_url):
    driver.get(detail_url)
    time.sleep(1.5)

    data = {}

    # 한글명 - DescriptionTitle div에서 "성분소개" 제거
    try:
        desc_title = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
        )
        data["한글명"] = desc_title.text.replace("성분소개", "").strip()
    except NoSuchElementException:
        data["한글명"] = ""

    # 영문명 - EnglishName 클래스 h3 태그
    try:
        data["영문명"] = driver.find_element(
            By.CSS_SELECTOR, "h3[class*='EnglishName']"
        ).text.strip()
    except NoSuchElementException:
        data["영문명"] = ""

    # 등급
    try:
        data["등급"] = driver.find_element(
            By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
        ).text.strip()
    except NoSuchElementException:
        data["등급"] = ""

    # 효과별 - Benefits 클래스 div 안의 a 태그들
    try:
        benefits_div = driver.find_element(By.CSS_SELECTOR, "div[class*='Benefits']")
        benefit_links = benefits_div.find_elements(By.TAG_NAME, "a")
        data["효과별"] = ", ".join(a.text.strip() for a in benefit_links if a.text.strip())
    except NoSuchElementException:
        data["효과별"] = ""

    # 분류 - "분류:" span의 부모 div 찾아서 그 안의 a 태그들 수집
    try:
        # "분류:" 텍스트를 포함한 div (span의 부모의 부모)
        category_div = driver.find_element(
            By.XPATH, "//span[contains(text(),'분류:')]/parent::div"
        )
        category_links = category_div.find_elements(By.TAG_NAME, "a")
        data["분류"] = ", ".join(a.text.strip() for a in category_links if a.text.strip())
    except NoSuchElementException:
        data["분류"] = ""

    # 성분설명 - DescriptionTitle 다음 형제 div
    try:
        desc_body = driver.find_element(
            By.XPATH, "//div[contains(@class,'DescriptionTitle')]/following-sibling::div[1]"
        )
        data["성분설명"] = desc_body.text.strip()
    except NoSuchElementException:
        data["성분설명"] = ""

    # 연관성분 - RelatedWrapper 안의 a 태그들
    try:
        related_div = driver.find_element(By.CSS_SELECTOR, "div[class*='RelatedWrapper']")
        related_links = related_div.find_elements(By.TAG_NAME, "a")
        data["연관성분"] = ", ".join(a.text.strip() for a in related_links if a.text.strip())
    except NoSuchElementException:
        data["연관성분"] = ""

    # 참고논문 - __Reference- 클래스 div들 줄바꿈으로 이어붙임
    try:
        ref_divs = driver.find_elements(By.CSS_SELECTOR, "div[class*='__Reference-']")
        data["참고논문"] = "\n".join(d.text.strip() for d in ref_divs if d.text.strip())
    except NoSuchElementException:
        data["참고논문"] = ""

    return data

print("함수 재정의 완료!")

함수 재정의 완료!


In [ ]:
# ── 단일 성분 재확인 ──────────────────────────────────────────
result = scrape_detail_page(test_url)

for k, v in result.items():
    print(f"{k}: {v[:80] if v else '(없음)'}")

한글명: (아몬드/보리지/아마/올리브)애씨드/글리세라이즈
영문명: Almond/Borage/Linseed/Olive Acids/Glycerides*
등급: 훌륭함
효과별: 수분 공급, 안티에이징
분류: (없음)
성분설명: 아몬드, 보리지, 아마, 올리브 등 무향의 식물성 오일에서 비롯된 트라이글리세라이드와 지방산의 혼합물. 이 혼합물은 피부에 수분을 공급하고 보충
연관성분: 아몬드오일, 보리지씨오일
참고논문: American Journal of Clinical Dermatology, February 2018, pages 103-117
Internati


In [ ]:
# ── 분류 셀렉터만 집중 탐색 ───────────────────────────────────
driver.get(test_url)
time.sleep(2)

# "분류" 텍스트 포함한 모든 요소 출력
els = driver.find_elements(By.XPATH, "//*[contains(text(), '분류')]")
for el in els:
    print(f"태그: <{el.tag_name}>")
    print(f"클래스: {el.get_attribute('class')}")
    print(f"텍스트: {repr(el.text[:100])}")
    print()

태그: <div>
클래스: large7 normalcase ko_style
텍스트: '분류: 유연제(연화제), 항산화제'



In [ ]:
def scrape_detail_page(detail_url):
    driver.get(detail_url)
    time.sleep(1.5)

    data = {}

    # 한글명
    try:
        desc_title = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
        )
        data["한글명"] = desc_title.text.replace("성분소개", "").strip()
    except NoSuchElementException:
        data["한글명"] = ""

    # 영문명
    try:
        data["영문명"] = driver.find_element(
            By.CSS_SELECTOR, "h3[class*='EnglishName']"
        ).text.strip()
    except NoSuchElementException:
        data["영문명"] = ""

    # 등급
    try:
        data["등급"] = driver.find_element(
            By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
        ).text.strip()
    except NoSuchElementException:
        data["등급"] = ""

    # 효과별
    try:
        benefits_div = driver.find_element(By.CSS_SELECTOR, "div[class*='Benefits']")
        benefit_links = benefits_div.find_elements(By.TAG_NAME, "a")
        data["효과별"] = ", ".join(a.text.strip() for a in benefit_links if a.text.strip())
    except NoSuchElementException:
        data["효과별"] = ""

    # 분류 - "분류:"로 시작하는 div 텍스트에서 직접 추출
    try:
        category_div = driver.find_element(
            By.XPATH, "//div[starts-with(normalize-space(.), '분류:')]"
        )
        data["분류"] = category_div.text.replace("분류:", "").strip()
    except NoSuchElementException:
        data["분류"] = ""

    # 성분설명
    try:
        desc_body = driver.find_element(
            By.XPATH, "//div[contains(@class,'DescriptionTitle')]/following-sibling::div[1]"
        )
        data["성분설명"] = desc_body.text.strip()
    except NoSuchElementException:
        data["성분설명"] = ""

    # 연관성분
    try:
        related_div = driver.find_element(By.CSS_SELECTOR, "div[class*='RelatedWrapper']")
        related_links = related_div.find_elements(By.TAG_NAME, "a")
        data["연관성분"] = ", ".join(a.text.strip() for a in related_links if a.text.strip())
    except NoSuchElementException:
        data["연관성분"] = ""

    # 참고논문
    try:
        ref_divs = driver.find_elements(By.CSS_SELECTOR, "div[class*='__Reference-']")
        data["참고논문"] = "\n".join(d.text.strip() for d in ref_divs if d.text.strip())
    except NoSuchElementException:
        data["참고논문"] = ""

    return data

print("함수 재정의 완료!")

함수 재정의 완료!


In [ ]:
# ── 단일 성분 재확인 ──────────────────────────────────────────
result = scrape_detail_page(test_url)

for k, v in result.items():
    print(f"{k}: {v[:100] if v else '(없음)'}")

한글명: (아몬드/보리지/아마/올리브)애씨드/글리세라이즈
영문명: Almond/Borage/Linseed/Olive Acids/Glycerides*
등급: 훌륭함
효과별: 수분 공급, 안티에이징
분류: 유연제(연화제), 항산화제
성분설명: 아몬드, 보리지, 아마, 올리브 등 무향의 식물성 오일에서 비롯된 트라이글리세라이드와 지방산의 혼합물. 이 혼합물은 피부에 수분을 공급하고 보충하며 항산화 효과를 제공하고 피부 본
연관성분: 아몬드오일, 보리지씨오일
참고논문: American Journal of Clinical Dermatology, February 2018, pages 103-117
International Journal of Mole


In [ ]:
# 성분설명 전체 출력 확인
print(result["성분설명"])
print("\n---")
print(result["참고논문"])

아몬드, 보리지, 아마, 올리브 등 무향의 식물성 오일에서 비롯된 트라이글리세라이드와 지방산의 혼합물. 이 혼합물은 피부에 수분을 공급하고 보충하며 항산화 효과를 제공하고 피부 본연의 회복력을 향상할 수 있다.
*대한화장품성분사전에 등록된 표준화 국문, 영문명칭

---
American Journal of Clinical Dermatology, February 2018, pages 103-117
International Journal of Molecular Sciences, December 2017, ePublication
Journal of Cosmetic Dermatology, December 2016, pages 549-558


In [ ]:
# ── 10개 수집 ─────────────────────────────────────────────────
results = []

list_url = (
    "https://www.paulaschoice.co.kr/ingredients"
    "?csortb1=name&csortd1=1"
    "&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8"
    "&sz=10&start=0"
)

driver.get(list_url)
time.sleep(2)

ingredient_links = get_ingredient_links()
print(f"링크 {len(ingredient_links)}개 수집\n")

for idx, link in enumerate(ingredient_links):
    print(f"[{idx+1}/10] 수집 중...")
    detail_data = scrape_detail_page(link)
    results.append(detail_data)
    driver.get(list_url)
    time.sleep(1.5)

print("\n── 수집 결과 ──")
for i, r in enumerate(results):
    print(f"\n[{i+1}번째 성분]")
    for k, v in r.items():
        print(f"  {k}: {v if v else '(없음)'}")


링크 10개 수집

[1/10] 수집 중...
[2/10] 수집 중...
[3/10] 수집 중...
[4/10] 수집 중...
[5/10] 수집 중...
[6/10] 수집 중...
[7/10] 수집 중...
[8/10] 수집 중...
[9/10] 수집 중...
[10/10] 수집 중...

── 수집 결과 ──

[1번째 성분]
  한글명: (아몬드/보리지/아마/올리브)애씨드/글리세라이즈
  영문명: Almond/Borage/Linseed/Olive Acids/Glycerides*
  등급: 훌륭함
  효과별: 수분 공급, 안티에이징
  분류: 유연제(연화제), 항산화제
  성분설명: 아몬드, 보리지, 아마, 올리브 등 무향의 식물성 오일에서 비롯된 트라이글리세라이드와 지방산의 혼합물. 이 혼합물은 피부에 수분을 공급하고 보충하며 항산화 효과를 제공하고 피부 본연의 회복력을 향상할 수 있다.
*대한화장품성분사전에 등록된 표준화 국문, 영문명칭
  연관성분: 아몬드오일, 보리지씨오일
  참고논문: American Journal of Clinical Dermatology, February 2018, pages 103-117
International Journal of Molecular Sciences, December 2017, ePublication
Journal of Cosmetic Dermatology, December 2016, pages 549-558

[2번째 성분]
  한글명: 10-하이드록시데카노익애씨드
  영문명: 10-Hydroxydecanoic Acid*
  등급: 훌륭함
  효과별: 수분 공급, 안티에이징, 모공 크기 개선, 진정
  분류: 유연제(연화제), 수분증발차단제-불투명화제
  성분설명: 10-하이드록시데카노익애씨드는 벌집에서 일벌이 여왕벌과 애벌레에게 먹이를 주기 위해 생산하는 영양 물질인 로열젤리에서 추출할 수 있는 중간 사슬 지방산이다. 로열젤리 자체는 고대부터 약용으로 사용되어 왔지만, 10-하이드록시데카노익애씨드의 효능에 대

In [ ]:
import csv

output_file = "paulaschoice_ingredients.csv"

with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    # utf-8-sig : 엑셀에서 한글 깨짐 방지
    fieldnames = ["한글명", "영문명", "등급", "효과별", "분류", "성분설명", "연관성분", "참고논문"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()   # 첫 줄에 컬럼명 작성
    writer.writerows(results)  # 수집한 데이터 전체 작성

print(f"저장 완료! ({len(results)}개 성분)")

저장 완료! (10개 성분)


In [ ]:
import time
import csv
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

# ── 전체 결과 저장 리스트 ─────────────────────────────────────
results = []

# ── URL 템플릿 (start 값만 바꿔서 페이지 이동) ────────────────
BASE_URL = (
    "https://www.paulaschoice.co.kr/ingredients"
    "?csortb1=name&csortd1=1"
    "&crefn1=ingredientRating&crefv1=%ED%9B%8C%EB%A5%AD%ED%95%A8"
    "&sz=10&start={start}"
)


def get_ingredient_links():
    """현재 목록 페이지에서 성분 상세 링크 수집 (중복 제거)"""
    links = driver.find_elements(
        By.CSS_SELECTOR, "a[href*='/ingredients/ingredient-']"
    )
    seen = set()
    unique = []
    for el in links:
        href = el.get_attribute("href")
        if href and href not in seen:
            seen.add(href)
            unique.append(href)
    return unique


def scrape_detail_page(detail_url):
    """성분 상세 페이지에서 모든 필드 수집"""
    driver.get(detail_url)
    time.sleep(1.5)

    data = {}

    # 한글명
    try:
        desc_title = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
        )
        data["한글명"] = desc_title.text.replace("성분소개", "").strip()
    except NoSuchElementException:
        data["한글명"] = ""

    # 영문명
    try:
        data["영문명"] = driver.find_element(
            By.CSS_SELECTOR, "h3[class*='EnglishName']"
        ).text.strip()
    except NoSuchElementException:
        data["영문명"] = ""

    # 등급
    try:
        data["등급"] = driver.find_element(
            By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
        ).text.strip()
    except NoSuchElementException:
        data["등급"] = ""

    # 효과별
    try:
        benefits_div = driver.find_element(By.CSS_SELECTOR, "div[class*='Benefits']")
        benefit_links = benefits_div.find_elements(By.TAG_NAME, "a")
        data["효과별"] = ", ".join(a.text.strip() for a in benefit_links if a.text.strip())
    except NoSuchElementException:
        data["효과별"] = ""

    # 분류
    try:
        category_div = driver.find_element(
            By.XPATH, "//div[starts-with(normalize-space(.), '분류:')]"
        )
        data["분류"] = category_div.text.replace("분류:", "").strip()
    except NoSuchElementException:
        data["분류"] = ""

    # 성분설명
    try:
        desc_body = driver.find_element(
            By.XPATH, "//div[contains(@class,'DescriptionTitle')]/following-sibling::div[1]"
        )
        data["성분설명"] = desc_body.text.strip()
    except NoSuchElementException:
        data["성분설명"] = ""

    # 연관성분
    try:
        related_div = driver.find_element(By.CSS_SELECTOR, "div[class*='RelatedWrapper']")
        related_links = related_div.find_elements(By.TAG_NAME, "a")
        data["연관성분"] = ", ".join(a.text.strip() for a in related_links if a.text.strip())
    except NoSuchElementException:
        data["연관성분"] = ""

    # 참고논문
    try:
        ref_divs = driver.find_elements(By.CSS_SELECTOR, "div[class*='__Reference-']")
        data["참고논문"] = "\n".join(d.text.strip() for d in ref_divs if d.text.strip())
    except NoSuchElementException:
        data["참고논문"] = ""

    return data


# ── 메인 크롤링 루프 ──────────────────────────────────────────
current_start = 0
page_num = 1

while True:
    list_url = BASE_URL.format(start=current_start)
    print(f"\n[페이지 {page_num}] 로딩 중... (start={current_start})")

    driver.get(list_url)
    time.sleep(2)

    ingredient_links = get_ingredient_links()
    print(f"  → 링크 {len(ingredient_links)}개 발견")

    # 링크 0개면 마지막 페이지 지남 → 종료
    if not ingredient_links:
        print("  → 더 이상 성분 없음. 크롤링 종료.")
        break

    for idx, link in enumerate(ingredient_links):
        print(f"  [{idx+1}/{len(ingredient_links)}] {link.split('/')[-1][:40]}")

        try:
            detail_data = scrape_detail_page(link)
            results.append(detail_data)
        except Exception as e:
            # 특정 성분 수집 실패해도 전체 중단 없이 계속 진행
            print(f"    !! 수집 실패: {e}")
            results.append({"한글명": "수집실패", "영문명": link, "등급": "",
                            "효과별": "", "분류": "", "성분설명": "",
                            "연관성분": "", "참고논문": ""})

        # 목록 페이지로 복귀
        driver.get(list_url)
        time.sleep(1.5)

    # 중간 저장 (페이지마다 저장해서 중간에 끊겨도 데이터 보존)
    output_file = "paulaschoice_ingredients.csv"
    with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
        fieldnames = ["한글명", "영문명", "등급", "효과별", "분류", "성분설명", "연관성분", "참고논문"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"  → 중간 저장 완료 (누적 {len(results)}개)")

    # 다음 페이지
    current_start += 10
    page_num += 1


print(f"\n크롤링 완료! 총 {len(results)}개 성분 수집")


[페이지 1] 로딩 중... (start=0)
  → 링크 10개 발견
  [1/10] ingredient-almond-borage-linseed-olive-a
  [2/10] ingredient-10-hydroxydecanoic-acid.html?
  [3/10] ingredient-3-o-ethyl-ascorbic-acid.html?
  [4/10] ingredient-c10-30-cholesterol-lanosterol
  [5/10] ingredient-c-13-16-isoalkane.html?csortb
  [6/10] ingredient-d-alpha-tocopherol.html?csort
  [7/10] ingredient-d-panthenol.html?csortb1=name
  [8/10] ingredient-ester-c.html?csortb1=name&cso
  [9/10] ingredient-l-ascorbic-acid.html?csortb1=
  [10/10] ingredient-n-hydroxysuccinimide.html?cso
  → 중간 저장 완료 (누적 10개)

[페이지 2] 로딩 중... (start=10)
  → 링크 10개 발견
  [1/10] ingredient-syn-tc.html?csortb1=name&csor
  [2/10] ingredient-polygonum-bistorta-root-extra
  [3/10] ingredient-prickly-pear.html?csortb1=nam
  [4/10] ingredient-terminalia-chebula-fruit-extr
  [5/10] ingredient-gallic-acid.html?csortb1=name
  [6/10] ingredient-gamma-linolenic-acid-gla.html
  [7/10] ingredient-licorice-root.html?csortb1=na
  [8/10] ingredient-glycyrrhiza-uralensis-ro

In [ ]:
# ── 최종 CSV 저장 및 확인 ─────────────────────────────────────
output_file = "paulaschoice_ingredients.csv"

with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    fieldnames = ["한글명", "영문명", "등급", "효과별", "분류", "성분설명", "연관성분", "참고논문"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"최종 저장 완료 → {output_file} ({len(results)}개)")

# 데이터프레임으로 확인
df = pd.read_csv(output_file, encoding="utf-8-sig")
print(f"컬럼: {list(df.columns)}")
print(f"행 수: {len(df)}")
df.head()

최종 저장 완료 → paulaschoice_ingredients.csv (842개)
컬럼: ['한글명', '영문명', '등급', '효과별', '분류', '성분설명', '연관성분', '참고논문']
행 수: 842


,한글명,영문명,등급,효과별,분류,성분설명,연관성분,참고논문
0,(아몬드/보리지/아마/올리브)애씨드/글리세라이즈,Almond/Borage/Linseed/Olive Acids/Glycerides*,훌륭함,"수분 공급, 안티에이징","유연제(연화제), 항산화제","아몬드, 보리지, 아마, 올리브 등 무향의 식물성 오일에서 비롯된 트라이글리세라이드...","아몬드오일, 보리지씨오일","American Journal of Clinical Dermatology, Febr..."
1,10-하이드록시데카노익애씨드,10-Hydroxydecanoic Acid*,훌륭함,"수분 공급, 안티에이징, 모공 크기 개선, 진정","유연제(연화제), 수분증발차단제-불투명화제",10-하이드록시데카노익애씨드는 벌집에서 일벌이 여왕벌과 애벌레에게 먹이를 주기 위해...,"로얄젤리, 지방산","Cosmetic Ingredient Review, Website, Accessed ..."
2,3-O-에틸아스코빅애씨드,3-O-Ethyl Ascorbic Acid*,훌륭함,"안티에이징, 피부 톤 개선, 다크스팟 개선",항산화제,3-O-에틸아스코빅애씨드 순수한 비타민 C (아스코빅애씨드)의 유도체로 안정적이며 ...,"아스코빅애씨드, 비타민 C","Free Radical Biology and Medicine, September 2..."
3,C10-30콜레스테롤/라노스테롤에스터,C10-30 Cholesterol/Lanosterol Esters*,훌륭함,수분 공급,"유연제(연화제), 질감향상제","라놀린과 콜레스테롤에서 유래된 지방산의 복합 혼합물이며, 피부 상태를 조절하고 화장...",NaN,"Journal of Clinical and Aesthetic Dermatology,..."
4,C13-16아이소알케인,C13-16 Isoalkane*,훌륭함,NaN,NaN,C13-16아이소알케인은 식물에서 추출할 수 있는 합성 용제이다. (폴라초이스는 식...,C13-14아이소알케인,"International Journal of Toxicology, November-..."


In [ ]:
import time
import csv
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

results = []

BASE_URL = (
    "https://www.paulaschoice.co.kr/ingredients"
    "?crefn1=ingredientRating&crefv1=%EC%A2%8B%EC%9D%8C"
    "&csortb1=name&csortd1=1"
    "&sz=10&start={start}"
)


def get_ingredient_links():
    links = driver.find_elements(
        By.CSS_SELECTOR, "a[href*='/ingredients/ingredient-']"
    )
    seen = set()
    unique = []
    for el in links:
        href = el.get_attribute("href")
        if href and href not in seen:
            seen.add(href)
            unique.append(href)
    return unique


def scrape_detail_page(detail_url):
    driver.get(detail_url)
    time.sleep(1.5)

    data = {}

    try:
        desc_title = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
        )
        data["한글명"] = desc_title.text.replace("성분소개", "").strip()
    except NoSuchElementException:
        data["한글명"] = ""

    try:
        data["영문명"] = driver.find_element(
            By.CSS_SELECTOR, "h3[class*='EnglishName']"
        ).text.strip()
    except NoSuchElementException:
        data["영문명"] = ""

    try:
        data["등급"] = driver.find_element(
            By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
        ).text.strip()
    except NoSuchElementException:
        data["등급"] = ""

    try:
        benefits_div = driver.find_element(By.CSS_SELECTOR, "div[class*='Benefits']")
        benefit_links = benefits_div.find_elements(By.TAG_NAME, "a")
        data["효과별"] = ", ".join(a.text.strip() for a in benefit_links if a.text.strip())
    except NoSuchElementException:
        data["효과별"] = ""

    try:
        category_div = driver.find_element(
            By.XPATH, "//div[starts-with(normalize-space(.), '분류:')]"
        )
        data["분류"] = category_div.text.replace("분류:", "").strip()
    except NoSuchElementException:
        data["분류"] = ""

    try:
        desc_body = driver.find_element(
            By.XPATH, "//div[contains(@class,'DescriptionTitle')]/following-sibling::div[1]"
        )
        data["성분설명"] = desc_body.text.strip()
    except NoSuchElementException:
        data["성분설명"] = ""

    try:
        related_div = driver.find_element(By.CSS_SELECTOR, "div[class*='RelatedWrapper']")
        related_links = related_div.find_elements(By.TAG_NAME, "a")
        data["연관성분"] = ", ".join(a.text.strip() for a in related_links if a.text.strip())
    except NoSuchElementException:
        data["연관성분"] = ""

    try:
        ref_divs = driver.find_elements(By.CSS_SELECTOR, "div[class*='__Reference-']")
        data["참고논문"] = "\n".join(d.text.strip() for d in ref_divs if d.text.strip())
    except NoSuchElementException:
        data["참고논문"] = ""

    return data


# ── 메인 크롤링 루프 ──────────────────────────────────────────
current_start = 0
page_num = 1
output_file = "paulaschoice_good_ingredients.csv"

while True:
    list_url = BASE_URL.format(start=current_start)
    print(f"\n[페이지 {page_num}] 로딩 중... (start={current_start})")

    driver.get(list_url)
    time.sleep(2)

    ingredient_links = get_ingredient_links()
    print(f"  → 링크 {len(ingredient_links)}개 발견")

    if not ingredient_links:
        print("  → 더 이상 성분 없음. 크롤링 종료.")
        break

    for idx, link in enumerate(ingredient_links):
        print(f"  [{idx+1}/{len(ingredient_links)}] {link.split('/')[-1][:40]}")

        try:
            detail_data = scrape_detail_page(link)
            results.append(detail_data)
        except Exception as e:
            print(f"    !! 수집 실패: {e}")
            results.append({
                "한글명": "수집실패", "영문명": link, "등급": "",
                "효과별": "", "분류": "", "성분설명": "",
                "연관성분": "", "참고논문": ""
            })

        driver.get(list_url)
        time.sleep(1.5)

    # 페이지마다 중간 저장
    with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
        fieldnames = ["한글명", "영문명", "등급", "효과별", "분류", "성분설명", "연관성분", "참고논문"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"  → 중간 저장 완료 (누적 {len(results)}개)")

    current_start += 10
    page_num += 1

print(f"\n크롤링 완료! 총 {len(results)}개 성분 수집")


[페이지 1] 로딩 중... (start=0)
  → 링크 10개 발견
  [1/10] ingredient-1-2-hexanediol.html?csortb1=n
  [2/10] ingredient-2-3-butanediol.html?csortb1=n
  [3/10] ingredient-4-t-butylcyclohexanol.html?cs
  [4/10] ingredient-quaternary-ammonium-compounds
  [5/10] ingredient-c11-15-pareth-7.html?csortb1=
  [6/10] ingredient-c12-15-alkyl-benzoate.html?cs
  [7/10] ingredient-c12-16-pareth-9.html?csortb1=
  [8/10] ingredient-c12-18-acid-triglyceride.html
  [9/10] ingredient-c13-14-isoalkane.html?csortb1
  [10/10] ingredient-c13-14-isoparaffin.html?csort
  → 중간 저장 완료 (누적 10개)

[페이지 2] 로딩 중... (start=10)
  → 링크 10개 발견
  [1/10] ingredient-c13-15-alkane.html?csortb1=na
  [2/10] ingredient-c14-22-alcohols.html?csortb1=
  [3/10] ingredient-c15-19-alkane.html?csortb1=na
  [4/10] ingredient-c18-36-acid-triglyceride.html
  [5/10] ingredient-c18-38-alkyl-hydroxystearoyl-
  [6/10] ingredient-c20-40-alketh-40.html?csortb1
  [7/10] ingredient-c20-40-pareth-40.html?csortb1
  [8/10] ingredient-c9-12-alkane.html?csortb

In [ ]:
import time
import csv
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

results = []

BASE_URL = (
    "https://www.paulaschoice.co.kr/ingredients"
    "?crefn1=ingredientRating&crefv1=%EB%B3%B4%ED%86%B5"
    "&csortb1=name&csortd1=1"
    "&sz=10&start={start}"
)


def get_ingredient_links():
    links = driver.find_elements(
        By.CSS_SELECTOR, "a[href*='/ingredients/ingredient-']"
    )
    seen = set()
    unique = []
    for el in links:
        href = el.get_attribute("href")
        if href and href not in seen:
            seen.add(href)
            unique.append(href)
    return unique


def scrape_detail_page(detail_url):
    driver.get(detail_url)
    time.sleep(1.5)

    data = {}

    try:
        desc_title = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
        )
        data["한글명"] = desc_title.text.replace("성분소개", "").strip()
    except NoSuchElementException:
        data["한글명"] = ""

    try:
        data["영문명"] = driver.find_element(
            By.CSS_SELECTOR, "h3[class*='EnglishName']"
        ).text.strip()
    except NoSuchElementException:
        data["영문명"] = ""

    try:
        data["등급"] = driver.find_element(
            By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
        ).text.strip()
    except NoSuchElementException:
        data["등급"] = ""

    try:
        benefits_div = driver.find_element(By.CSS_SELECTOR, "div[class*='Benefits']")
        benefit_links = benefits_div.find_elements(By.TAG_NAME, "a")
        data["효과별"] = ", ".join(a.text.strip() for a in benefit_links if a.text.strip())
    except NoSuchElementException:
        data["효과별"] = ""

    try:
        category_div = driver.find_element(
            By.XPATH, "//div[starts-with(normalize-space(.), '분류:')]"
        )
        data["분류"] = category_div.text.replace("분류:", "").strip()
    except NoSuchElementException:
        data["분류"] = ""

    try:
        desc_body = driver.find_element(
            By.XPATH, "//div[contains(@class,'DescriptionTitle')]/following-sibling::div[1]"
        )
        data["성분설명"] = desc_body.text.strip()
    except NoSuchElementException:
        data["성분설명"] = ""

    try:
        related_div = driver.find_element(By.CSS_SELECTOR, "div[class*='RelatedWrapper']")
        related_links = related_div.find_elements(By.TAG_NAME, "a")
        data["연관성분"] = ", ".join(a.text.strip() for a in related_links if a.text.strip())
    except NoSuchElementException:
        data["연관성분"] = ""

    try:
        ref_divs = driver.find_elements(By.CSS_SELECTOR, "div[class*='__Reference-']")
        data["참고논문"] = "\n".join(d.text.strip() for d in ref_divs if d.text.strip())
    except NoSuchElementException:
        data["참고논문"] = ""

    return data


# ── 메인 크롤링 루프 ──────────────────────────────────────────
current_start = 0
page_num = 1
output_file = "paulaschoice_normal_ingredients.csv"  # 보통 등급

while True:
    list_url = BASE_URL.format(start=current_start)
    print(f"\n[페이지 {page_num}] 로딩 중... (start={current_start})")

    driver.get(list_url)
    time.sleep(2)

    ingredient_links = get_ingredient_links()
    print(f"  → 링크 {len(ingredient_links)}개 발견")

    if not ingredient_links:
        print("  → 더 이상 성분 없음. 크롤링 종료.")
        break

    for idx, link in enumerate(ingredient_links):
        print(f"  [{idx+1}/{len(ingredient_links)}] {link.split('/')[-1][:40]}")

        try:
            detail_data = scrape_detail_page(link)
            results.append(detail_data)
        except Exception as e:
            print(f"    !! 수집 실패: {e}")
            results.append({
                "한글명": "수집실패", "영문명": link, "등급": "",
                "효과별": "", "분류": "", "성분설명": "",
                "연관성분": "", "참고논문": ""
            })

        driver.get(list_url)
        time.sleep(1.5)

    with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
        fieldnames = ["한글명", "영문명", "등급", "효과별", "분류", "성분설명", "연관성분", "참고논문"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"  → 중간 저장 완료 (누적 {len(results)}개)")

    current_start += 10
    page_num += 1

print(f"\n크롤링 완료! 총 {len(results)}개 성분 수집")


[페이지 1] 로딩 중... (start=0)
  → 링크 10개 발견
  [1/10] ingredient-c11-15-alkane-cycloalkane.htm
  [2/10] ingredient-dc.html?csortb1=name&csortd1=
  [3/10] ingredient-dc-colors.html?csortb1=name&c
  [4/10] ingredient-ext-dc.html?csortb1=name&csor
  [5/10] ingredient-fdc.html?csortb1=name&csortd1
  [6/10] ingredient-fdc-colors.html?csortb1=name&
  [7/10] ingredient-n-acetyl-l-tyrosine.html?csor
  [8/10] ingredient-plasticizing-agents.html?csor
  [9/10] ingredient-diospyros-kaki-fruit-extract.
  [10/10] ingredient-coriander.html?csortb1=name&c
  → 중간 저장 완료 (누적 10개)

[페이지 2] 로딩 중... (start=10)
  → 링크 10개 발견
  [1/10] ingredient-fruit-acid.html?csortb1=name&
  [2/10] ingredient-citrus-unshiu.html?csortb1=na
  [3/10] ingredient-citrus-unshiu-peel-extract.ht
  [4/10] ingredient-panicum-miliaceum.html?csortb
  [5/10] ingredient-panicum-millaceum-millet-seed
  [6/10] ingredient-millet-seed-extract.html?csor
  [7/10] ingredient-nylon-12-fluorescent-brighten
  [8/10] ingredient-nettle-extract.html?csor

In [ ]:
import time
import csv
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

results = []

BASE_URL = (
    "https://www.paulaschoice.co.kr/ingredients"
    "?crefn1=ingredientRating&crefv1=%EB%82%98%EC%81%A8"
    "&csortb1=name&csortd1=1"
    "&sz=10&start={start}"
)


def get_ingredient_links():
    links = driver.find_elements(
        By.CSS_SELECTOR, "a[href*='/ingredients/ingredient-']"
    )
    seen = set()
    unique = []
    for el in links:
        href = el.get_attribute("href")
        if href and href not in seen:
            seen.add(href)
            unique.append(href)
    return unique


def scrape_detail_page(detail_url):
    driver.get(detail_url)
    time.sleep(1.5)

    data = {}

    try:
        desc_title = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
        )
        data["한글명"] = desc_title.text.replace("성분소개", "").strip()
    except NoSuchElementException:
        data["한글명"] = ""

    try:
        data["영문명"] = driver.find_element(
            By.CSS_SELECTOR, "h3[class*='EnglishName']"
        ).text.strip()
    except NoSuchElementException:
        data["영문명"] = ""

    try:
        data["등급"] = driver.find_element(
            By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
        ).text.strip()
    except NoSuchElementException:
        data["등급"] = ""

    try:
        benefits_div = driver.find_element(By.CSS_SELECTOR, "div[class*='Benefits']")
        benefit_links = benefits_div.find_elements(By.TAG_NAME, "a")
        data["효과별"] = ", ".join(a.text.strip() for a in benefit_links if a.text.strip())
    except NoSuchElementException:
        data["효과별"] = ""

    try:
        category_div = driver.find_element(
            By.XPATH, "//div[starts-with(normalize-space(.), '분류:')]"
        )
        data["분류"] = category_div.text.replace("분류:", "").strip()
    except NoSuchElementException:
        data["분류"] = ""

    try:
        desc_body = driver.find_element(
            By.XPATH, "//div[contains(@class,'DescriptionTitle')]/following-sibling::div[1]"
        )
        data["성분설명"] = desc_body.text.strip()
    except NoSuchElementException:
        data["성분설명"] = ""

    try:
        related_div = driver.find_element(By.CSS_SELECTOR, "div[class*='RelatedWrapper']")
        related_links = related_div.find_elements(By.TAG_NAME, "a")
        data["연관성분"] = ", ".join(a.text.strip() for a in related_links if a.text.strip())
    except NoSuchElementException:
        data["연관성분"] = ""

    try:
        ref_divs = driver.find_elements(By.CSS_SELECTOR, "div[class*='__Reference-']")
        data["참고논문"] = "\n".join(d.text.strip() for d in ref_divs if d.text.strip())
    except NoSuchElementException:
        data["참고논문"] = ""

    return data


# ── 메인 크롤링 루프 ──────────────────────────────────────────
current_start = 0
page_num = 1
output_file = "paulaschoice_bad_ingredients.csv"  # 나쁨 등급

while True:
    list_url = BASE_URL.format(start=current_start)
    print(f"\n[페이지 {page_num}] 로딩 중... (start={current_start})")

    driver.get(list_url)
    time.sleep(2)

    ingredient_links = get_ingredient_links()
    print(f"  → 링크 {len(ingredient_links)}개 발견")

    if not ingredient_links:
        print("  → 더 이상 성분 없음. 크롤링 종료.")
        break

    for idx, link in enumerate(ingredient_links):
        print(f"  [{idx+1}/{len(ingredient_links)}] {link.split('/')[-1][:40]}")

        try:
            detail_data = scrape_detail_page(link)
            results.append(detail_data)
        except Exception as e:
            print(f"    !! 수집 실패: {e}")
            results.append({
                "한글명": "수집실패", "영문명": link, "등급": "",
                "효과별": "", "분류": "", "성분설명": "",
                "연관성분": "", "참고논문": ""
            })

        driver.get(list_url)
        time.sleep(1.5)

    with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
        fieldnames = ["한글명", "영문명", "등급", "효과별", "분류", "성분설명", "연관성분", "참고논문"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"  → 중간 저장 완료 (누적 {len(results)}개)")

    current_start += 10
    page_num += 1

print(f"\n크롤링 완료! 총 {len(results)}개 성분 수집")


[페이지 1] 로딩 중... (start=0)
  → 링크 10개 발견
  [1/10] ingredient-paullinia-cupana-seed-extract
  [2/10] ingredient-diatomaceous-earth.html?csort
  [3/10] ingredient-neroli-oil.html?csortb1=name&
  [4/10] ingredient-neem-oil.html?csortb1=name&cs
  [5/10] ingredient-rosa-damascena-flower-water.h
  [6/10] ingredient-dimethyl-mea.html?csortb1=nam
  [7/10] ingredient-diamond-powder.html?csortb1=n
  [8/10] ingredient-dicalcium-phosphate.html?csor
  [9/10] ingredient-diazolidinyl-urea.html?csortb
  [10/10] ingredient-dmdm-hydantoin.html?csortb1=n
  → 중간 저장 완료 (누적 10개)

[페이지 2] 로딩 중... (start=10)
  → 링크 10개 발견
  [1/10] ingredient-dmae.html?csortb1=name&csortd
  [2/10] ingredient-myrtus-communis-extract.html?
  [3/10] ingredient-verbena-officinalis-leaf-extr
  [4/10] ingredient-myrtle-extract.html?csortb1=n
  [5/10] ingredient-methyldihydrojasmonate.html?c
  [6/10] ingredient-mea.html?csortb1=name&csortd1
  [7/10] ingredient-mimosa-oil-or-extract.html?cs
  [8/10] ingredient-vetiver-zizanoides-root-

In [ ]:
import time
import csv
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

results = []

BASE_URL = (
    "https://www.paulaschoice.co.kr/ingredients"
    "?crefn1=ingredientRating&crefv1=%EB%A7%A4%EC%9A%B0%20%EB%82%98%EC%81%A8"
    "&csortb1=name&csortd1=1"
    "&sz=10&start={start}"
)


def get_ingredient_links():
    links = driver.find_elements(
        By.CSS_SELECTOR, "a[href*='/ingredients/ingredient-']"
    )
    seen = set()
    unique = []
    for el in links:
        href = el.get_attribute("href")
        if href and href not in seen:
            seen.add(href)
            unique.append(href)
    return unique


def scrape_detail_page(detail_url):
    driver.get(detail_url)
    time.sleep(1.5)

    data = {}

    try:
        desc_title = driver.find_element(
            By.XPATH, "//div[contains(@class, 'DescriptionTitle')]"
        )
        data["한글명"] = desc_title.text.replace("성분소개", "").strip()
    except NoSuchElementException:
        data["한글명"] = ""

    try:
        data["영문명"] = driver.find_element(
            By.CSS_SELECTOR, "h3[class*='EnglishName']"
        ).text.strip()
    except NoSuchElementException:
        data["영문명"] = ""

    try:
        data["등급"] = driver.find_element(
            By.CSS_SELECTOR, "span.ColoredIngredientRating__Rating-sc-r02772-0"
        ).text.strip()
    except NoSuchElementException:
        data["등급"] = ""

    try:
        benefits_div = driver.find_element(By.CSS_SELECTOR, "div[class*='Benefits']")
        benefit_links = benefits_div.find_elements(By.TAG_NAME, "a")
        data["효과별"] = ", ".join(a.text.strip() for a in benefit_links if a.text.strip())
    except NoSuchElementException:
        data["효과별"] = ""

    try:
        category_div = driver.find_element(
            By.XPATH, "//div[starts-with(normalize-space(.), '분류:')]"
        )
        data["분류"] = category_div.text.replace("분류:", "").strip()
    except NoSuchElementException:
        data["분류"] = ""

    try:
        desc_body = driver.find_element(
            By.XPATH, "//div[contains(@class,'DescriptionTitle')]/following-sibling::div[1]"
        )
        data["성분설명"] = desc_body.text.strip()
    except NoSuchElementException:
        data["성분설명"] = ""

    try:
        related_div = driver.find_element(By.CSS_SELECTOR, "div[class*='RelatedWrapper']")
        related_links = related_div.find_elements(By.TAG_NAME, "a")
        data["연관성분"] = ", ".join(a.text.strip() for a in related_links if a.text.strip())
    except NoSuchElementException:
        data["연관성분"] = ""

    try:
        ref_divs = driver.find_elements(By.CSS_SELECTOR, "div[class*='__Reference-']")
        data["참고논문"] = "\n".join(d.text.strip() for d in ref_divs if d.text.strip())
    except NoSuchElementException:
        data["참고논문"] = ""

    return data


# ── 메인 크롤링 루프 ──────────────────────────────────────────
current_start = 0
page_num = 1
output_file = "paulaschoice_very_bad_ingredients.csv"  # 매우 나쁨 등급

while True:
    list_url = BASE_URL.format(start=current_start)
    print(f"\n[페이지 {page_num}] 로딩 중... (start={current_start})")

    driver.get(list_url)
    time.sleep(2)

    ingredient_links = get_ingredient_links()
    print(f"  → 링크 {len(ingredient_links)}개 발견")

    if not ingredient_links:
        print("  → 더 이상 성분 없음. 크롤링 종료.")
        break

    for idx, link in enumerate(ingredient_links):
        print(f"  [{idx+1}/{len(ingredient_links)}] {link.split('/')[-1][:40]}")

        try:
            detail_data = scrape_detail_page(link)
            results.append(detail_data)
        except Exception as e:
            print(f"    !! 수집 실패: {e}")
            results.append({
                "한글명": "수집실패", "영문명": link, "등급": "",
                "효과별": "", "분류": "", "성분설명": "",
                "연관성분": "", "참고논문": ""
            })

        driver.get(list_url)
        time.sleep(1.5)

    with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
        fieldnames = ["한글명", "영문명", "등급", "효과별", "분류", "성분설명", "연관성분", "참고논문"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"  → 중간 저장 완료 (누적 {len(results)}개)")

    current_start += 10
    page_num += 1

print(f"\n크롤링 완료! 총 {len(results)}개 성분 수집")


[페이지 1] 로딩 중... (start=0)
  → 링크 10개 발견
  [1/10] ingredient-bronopol.html?csortb1=name&cs
  [2/10] ingredient-sd-alcohol.html?csortb1=name&
  [3/10] ingredient-uva.html?csortb1=name&csortd1
  [4/10] ingredient-uvb.html?csortb1=name&csortd1
  [5/10] ingredient-lye.html?csortb1=name&csortd1
  [6/10] ingredient-galbanum.html?csortb1=name&cs
  [7/10] ingredient-ferula-galbaniflua.html?csort
  [8/10] ingredient-cinnamomum.html?csortb1=name&
  [9/10] ingredient-cinnamon.html?csortb1=name&cs
  [10/10] ingredient-capsicum-oleoresin.html?csort
  → 중간 저장 완료 (누적 10개)

[페이지 2] 로딩 중... (start=10)
  → 링크 10개 발견
  [1/10] ingredient-hydrastis-canadenis.html?csor
  [2/10] ingredient-guarana.html?csortb1=name&cso
  [3/10] ingredient-pogostemon-cablin.html?csortb
  [4/10] ingredient-photosensitizer.html?csortb1=
  [5/10] ingredient-orchid.html?csortb1=name&csor
  [6/10] ingredient-orchid-extract.html?csortb1=n
  [7/10] ingredient-neroli.html?csortb1=name&csor
  [8/10] ingredient-cinnamomum-camphora.html